# Chatterbox deepfake generation (Kaggle) — Swahili Common Voice

**Kaggle port of the Chatterbox variant.** Instead of re-downloading the
~20 GB Common Voice archive from Mozilla Data Collective, this notebook pulls
the reference audio a previous run already selected, preprocessed, and backed
up to Google Drive (`MyDrive/swahili-deepfake-dataset/data/`), then generates
**exactly one** Chatterbox Multilingual deepfake per real clip — same count,
no more, no less.

The full MDC download → select → preprocess pipeline is still here as an
**optional fallback** (Section 4), guarded by `DATA_SOURCE` in Section 0. Set
`DATA_SOURCE = "mdc"` only if the Drive copy is lost and you need to rebuild
the preprocessed set from scratch on Kaggle.

## Before running — Kaggle notebook settings (right-hand panel)
1. **Settings → Accelerator → GPU** (T4 or P100). Chatterbox needs a GPU.
2. **Settings → Internet → On** — git clone, pip, Google Drive (and, in
   fallback mode, MDC) all need it.
3. **Add-ons → Secrets**, add:
   - `RCLONE_CONF` — the full text of an `rclone.conf` that defines a Google
     Drive remote (see Section 2 for how to generate it).
   - `MDC_API_KEY` — only needed for the Section 4 fallback.

## One-time rclone setup on your own machine
```
rclone config
#   n) New remote      name> gdrive        storage> drive
#   scope> 1   (full access, so this notebook can also push results back;
#               use 3 = drive.readonly if you only ever want to pull)
#   ...complete the browser sign-in...
rclone config show gdrive
```
Paste everything from the `[gdrive]` line onward into the `RCLONE_CONF`
secret. The remote name (`gdrive`) must match `RCLONE_REMOTE` below.

## 0. Configuration

In [ ]:
# ---- Source of the real reference audio ----
#   "drive" : pull the already-preprocessed clips from Google Drive (default;
#             fast, no multi-GB download). Requires the RCLONE_CONF secret.
#   "mdc"   : rebuild everything from the Mozilla Data Collective archive
#             (Section 4). Only needed if the Drive copy is gone.
DATA_SOURCE = "drive"

# rclone remote name -- must match the [name] header inside your RCLONE_CONF.
RCLONE_REMOTE = "gdrive"
# Folder scripts/sync_drive.py backed up to, relative to the remote root (My Drive).
DRIVE_DATA_DIR = "swahili-deepfake-dataset/data"

# Push generated deepfakes + manifests back to Drive after each checkpoint.
# Requires the rclone remote to have write scope (not drive.readonly).
PUSH_RESULTS_TO_DRIVE = True

# Chatterbox Multilingual
DEVICE = "cuda"
LANGUAGE_ID = "sw"
CHECKPOINT_SIZE = 500        # generate in batches of this many, sync between

REPO_URL = "https://github.com/regak/Swahili-Deepfake-dataset.git"
REPO_REF = "main"   # branch/tag that contains the chatterbox + voxcpm generators and the tmp-file fix
REPO_DIR = "/kaggle/working/Swahili-Deepfake-dataset"
SCRATCH = "/kaggle/temp"     # not persisted; fine for smoke-test artefacts

RUN_MDC = (DATA_SOURCE == "mdc")
NEED_RCLONE = (DATA_SOURCE == "drive") or PUSH_RESULTS_TO_DRIVE

import os
os.makedirs(SCRATCH, exist_ok=True)
print(f"DATA_SOURCE={DATA_SOURCE}  RUN_MDC={RUN_MDC}  NEED_RCLONE={NEED_RCLONE}")

## 1. Clone the pipeline repo and install dependencies

In [ ]:
!git clone --depth 1 --branch {REPO_REF} {REPO_URL} {REPO_DIR}

import os
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

!pip install -q -r requirements.txt
!pip install -q chatterbox-tts

# chatterbox-tts can leave torch and torchvision mismatched, and numba needs numpy==1.26.4.
import subprocess, sys
t = subprocess.run([sys.executable, '-c', 'from chatterbox.mtl_tts import ChatterboxMultilingualTTS'], capture_output=True, text=True)
if t.returncode != 0:
    print('chatterbox import failed -> repairing torch stack:', t.stderr[-600:])
    !pip install -q --force-reinstall torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124
    !pip install -q 'numpy==1.26.4'
else:
    print('chatterbox imports OK')


In [ ]:
import csv, sys
# Python's csv module defaults to a 128 KB field cap; the pipeline's TSVs can
# legitimately exceed it. Same bump the upstream scripts do.
try:
    csv.field_size_limit(sys.maxsize)
except OverflowError:
    csv.field_size_limit(2**31 - 1)

## 2. Install rclone and load the Drive remote from Kaggle Secrets

Kaggle has no `google.colab.drive`, so this uses `rclone` against the same
`MyDrive/swahili-deepfake-dataset/data/` folder `scripts/sync_drive.py` writes
to on Colab. The static binary needs no `apt`; the remote definition comes
from the `RCLONE_CONF` secret so no token is ever written into a cell.

In [ ]:
if NEED_RCLONE:
    !curl -sL https://downloads.rclone.org/rclone-current-linux-amd64.zip -o {SCRATCH}/rclone.zip
    !cd {SCRATCH} && unzip -oq rclone.zip && cp rclone-*-linux-amd64/rclone /usr/local/bin/ && chmod 755 /usr/local/bin/rclone
    !rclone version | head -n 1

    from pathlib import Path
    from kaggle_secrets import UserSecretsClient

    conf_dir = Path.home() / ".config" / "rclone"
    conf_dir.mkdir(parents=True, exist_ok=True)
    import re
    _raw = UserSecretsClient().get_secret("RCLONE_CONF")
    # Kaggle's secret box flattens the config onto one line; rebuild one setting per line.
    _flat = " ".join(_raw.replace(chr(0xFEFF), "").split())
    _known = "type|client_id|client_secret|scope|token|team_drive|root_folder_id|service_account_file|service_account_credentials|auth_url|token_url"
    _parts = re.split("[ ](?=(?:" + _known + ")[ ]*=)", _flat)
    _conf = chr(10).join(_parts).strip() + chr(10)
    (conf_dir / "rclone.conf").write_text(_conf)
    # Diagnostics: setting NAMES only -- never print values (they contain the login token).
    _has_remote = _parts[0].strip() == "[" + RCLONE_REMOTE + "]"
    _names = [p.split("=")[0].strip() for p in _parts[1:]]
    print("config rebuilt: remote header ok =", _has_remote, "| settings found =", _names)
    assert _has_remote, "RCLONE_CONF does not start with the [gdrive] header"
    assert any(p.startswith("token") and len(p) > 60 for p in _parts), "RCLONE_CONF has no OAuth token"
    print("Wrote", conf_dir / "rclone.conf")

    # Sanity check: list the backed-up subfolders.
    !rclone lsd "{RCLONE_REMOTE}:{DRIVE_DATA_DIR}"
else:
    print("Skipping rclone (DATA_SOURCE='mdc' and PUSH_RESULTS_TO_DRIVE=False).")

## 3. Restore preprocessed reference audio + manifests from Drive

`data/processed/` holds the 16 kHz mono WAV reference clips (named
`<utterance-id-stem>.wav`) from the previous run's `preprocess_audio.py` step.
`data/generated/` (if present) is restored too, so an interrupted run resumes
instead of regenerating.

In [ ]:
from pathlib import Path

if DATA_SOURCE == "drive":
    Path("data/processed").mkdir(parents=True, exist_ok=True)
    Path("data/manifests").mkdir(parents=True, exist_ok=True)

    !rclone copy "{RCLONE_REMOTE}:{DRIVE_DATA_DIR}/processed"  data/processed  --transfers 32 --checkers 32 --fast-list --stats 15s
    !rclone copy "{RCLONE_REMOTE}:{DRIVE_DATA_DIR}/manifests" data/manifests --fast-list
    !rclone copy "{RCLONE_REMOTE}:{DRIVE_DATA_DIR}/generated/chatterbox" data/generated/chatterbox --transfers 32 --checkers 32 --fast-list --stats 15s || true

    print("\nRestored:")
    !find data/processed -type f | wc -l
    !ls data/manifests
else:
    print("DATA_SOURCE='mdc' -> skipping Drive restore; run Section 4 instead.")

## 4. (fallback only) Rebuild the preprocessed set from Mozilla Data Collective

Every cell here is a no-op unless `DATA_SOURCE = "mdc"`. This is the original
~20 GB download → phoneme-balanced selection → preprocessing pipeline, ported
to Kaggle Secrets. Only run it if the Drive copy in Section 3 is gone.

In [ ]:
if RUN_MDC:
    !pip install -q datacollective

In [ ]:
if RUN_MDC:
    import os
    from kaggle_secrets import UserSecretsClient
    os.environ["MDC_API_KEY"] = UserSecretsClient().get_secret("MDC_API_KEY")
    print("MDC_API_KEY loaded from Kaggle Secrets.")

In [ ]:
if RUN_MDC:
    from datacollective import download_dataset

    DATASET_ID = "cmqim4c1000tmnr07zq3vwhor"  # slug: common-voice-scripted-speech-26-0-swahil-0228b2f6
    DOWNLOAD_DIR = "/kaggle/working/mdc_common_voice_sw"

    dataset_path = download_dataset(DATASET_ID, download_directory=DOWNLOAD_DIR)
    print("Downloaded archive to:", dataset_path)

In [ ]:
if RUN_MDC:
    from datacollective import load_dataset

    try:
        dataset = load_dataset(DATASET_ID, download_directory=DOWNLOAD_DIR)
        print(dataset.shape)
        display(dataset.head())
    except RuntimeError as e:
        dataset = None
        print("No registered schema for this dataset yet -- extract dataset_path manually instead:")
        print(e)

In [ ]:
if RUN_MDC:
    assert dataset is not None, (
        "load_dataset() returned no DataFrame and this Kaggle port doesn't "
        "implement the manual tar-extraction fallback -- prefer DATA_SOURCE='drive'."
    )
    !mkdir -p data/manifests
    dataset.to_csv("data/manifests/mdc_export.csv", index=False)

    !python scripts/select_subset.py \
        data/manifests/mdc_export.csv \
        --schema mdc \
        --target-size 10000 --max-per-speaker 100 \
        --output data/manifests/selected_subset.tsv \
        --report data/manifests/phoneme_coverage_report.json

In [ ]:
if RUN_MDC:
    !python scripts/preprocess_audio.py \
        "/kaggle/working/mdc_common_voice_sw/common-voice-scripted-speech-26-0-swahil-0228b2f6/cv-corpus-26.0-2026-06-12/sw/clips" \
        data/processed \
        --subset data/manifests/selected_subset.tsv \
        --min-duration 2.0 --max-duration 20.0

In [ ]:
if RUN_MDC:
    import shutil, os
    # data/processed/ now holds standardized self-contained copies -- the raw
    # extracted corpus and the compressed archive can be freed.
    for p in [
        "/kaggle/working/mdc_common_voice_sw/common-voice-scripted-speech-26-0-swahil-0228b2f6",
        "/kaggle/working/mdc_common_voice_sw/common-voice-scripted-speech-26-0-swahil-0228b2f6.tar.gz",
    ]:
        if os.path.isdir(p):
            shutil.rmtree(p); print("removed dir", p)
        elif os.path.isfile(p):
            os.remove(p); print("removed file", p)
    !du -sh /kaggle/working/mdc_common_voice_sw/ 2>/dev/null || echo "(archive dir gone)"

## 5. Resolve the working subset

Runs in either mode. Intersects `selected_subset.tsv` with the reference clips
actually on disk and writes `selected_subset.present.tsv` — generation below
uses only that, so the deepfake count equals the real-clip count 1:1 (no
attempts against missing audio, no silent divergence).

In [ ]:
import csv
from pathlib import Path

SUBSET_TSV = "data/manifests/selected_subset.tsv"
PRESENT_SUBSET_TSV = "data/manifests/selected_subset.present.tsv"
PROCESSED_DIR = Path("data/processed")
AUDIO_EXT = ".wav"

processed_stems = {p.stem for p in PROCESSED_DIR.glob(f"*{AUDIO_EXT}")}
print(f"Reference clips in data/processed: {len(processed_stems)}")

with open(SUBSET_TSV, newline="", encoding="utf-8") as f:
    all_rows = list(csv.DictReader(f, delimiter="\t"))
present_rows = [r for r in all_rows if Path(r["id"]).stem in processed_stems]
print(f"selected_subset.tsv rows: {len(all_rows)}  |  with reference audio on disk: {len(present_rows)}")

with open(PRESENT_SUBSET_TSV, "w", newline="", encoding="utf-8") as fout:
    w = csv.DictWriter(fout, fieldnames=list(all_rows[0].keys()), delimiter="\t")
    w.writeheader()
    w.writerows(present_rows)

N_TARGET = len(present_rows)
print(f"\nWrote {PRESENT_SUBSET_TSV} -> {N_TARGET} Chatterbox deepfakes will be generated.")
assert N_TARGET > 0, "No reference audio matched selected_subset.tsv -- check Section 3 / Section 4."

## 7. Sanity checks + baseline (real-only) manifest

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
!nvidia-smi

In [ ]:
# Empty fake manifest (header only) -- lets build_manifest.py run before any
# deepfakes exist, as a real-data sanity check.
!mkdir -p data/manifests
import os
if not os.path.exists("data/manifests/fake_manifest.tsv"):
    with open("data/manifests/fake_manifest.tsv", "w") as f:
        f.write("id\tsource_utterance_id\tspeaker_id\tgenerator\taudio_path\n")

!python scripts/build_manifest.py \
    data/manifests/selected_subset.present.tsv data/manifests/fake_manifest.tsv \
    data/processed --output data/manifests/dataset_manifest.csv \
    --summary data/manifests/split_summary.json

In [ ]:
import pandas as pd
from pathlib import Path

manifest = pd.read_csv("data/manifests/dataset_manifest.csv")
sample = manifest.sample(min(10, len(manifest)), random_state=42)
sample["exists"] = sample["audio_path"].apply(lambda p: Path(p).is_file())
print(sample[["id", "label", "source", "speaker_id", "split", "exists"]].to_string(index=False))
print(f"\nAll sampled rows exist on disk: {sample['exists'].all()}")

In [ ]:
import json, os
p = "data/manifests/phoneme_coverage_report.json"
if os.path.exists(p):
    report = json.load(open(p))
    print(f"Selected: {report['num_selected']} utterances, {report['num_speakers']} speakers")
    print(f"Core phoneme coverage: {report['core_coverage_pct']:.1f}% "
          f"({report['core_covered']}/{report['core_total']})")
    print(f"Missing core phonemes: {report['missing_core_phonemes'] or 'none'}")
else:
    print("phoneme_coverage_report.json not restored -- skipping (not required).")

## 8. Smoke test — run this first

Confirm the generator loads, runs on GPU, and produces genuine-sounding
Swahili before launching the full run.

In [ ]:
!mkdir -p {SCRATCH}
!head -n 6 data/manifests/selected_subset.present.tsv > {SCRATCH}/smoke_subset.tsv

!python scripts/generate_deepfakes.py \
    {SCRATCH}/smoke_subset.tsv data/processed {SCRATCH}/smoke_generated \
    --generators chatterbox \
    --device {DEVICE} \
    --manifest {SCRATCH}/smoke_fake_manifest.tsv \
    --progress-every 1
assert get_ipython().user_ns.get('_exit_code', 0) == 0, 'generation failed -- see traceback above'


In [ ]:
from IPython.display import Audio, display
import glob

files = sorted(glob.glob(f"{SCRATCH}/smoke_generated/chatterbox/*.wav"))
print(files)
if files:
    display(Audio(files[0]))

## 9. Full generation, checkpointed every `CHECKPOINT_SIZE` utterances

Growing prefixes of `selected_subset.present.tsv` — already-done rows are
skipped near-instantly on a re-run — with an rclone push to Drive after each
checkpoint, so a Kaggle timeout never loses more than the last batch.

In [ ]:
import math

with open("data/manifests/selected_subset.present.tsv", newline="", encoding="utf-8") as f:
    header = f.readline()
    lines = f.readlines()

total = len(lines)
num_checkpoints = math.ceil(total / CHECKPOINT_SIZE)
print(f"{total} utterances -> {num_checkpoints} checkpoint(s) of {CHECKPOINT_SIZE}")

for i in range(1, num_checkpoints + 1):
    cutoff = min(i * CHECKPOINT_SIZE, total)
    prefix_path = f"{SCRATCH}/subset_prefix.tsv"
    with open(prefix_path, "w", encoding="utf-8") as pf:
        pf.write(header)
        pf.writelines(lines[:cutoff])

    print(f"--- Checkpoint {i}/{num_checkpoints}: utterances 1-{cutoff} of {total} ---")
    !python scripts/generate_deepfakes.py {prefix_path} data/processed data/generated \
        --generators chatterbox --device {DEVICE} --progress-every 10 \
        --manifest data/manifests/fake_manifest_chatterbox.tsv

    if PUSH_RESULTS_TO_DRIVE:
        print(f"Pushing to Drive after {cutoff} utterances...")
        !rclone copy data/generated/chatterbox "{RCLONE_REMOTE}:{DRIVE_DATA_DIR}/generated/chatterbox" --transfers 32 --checkers 32 --fast-list
        !rclone copy data/manifests "{RCLONE_REMOTE}:{DRIVE_DATA_DIR}/manifests" --fast-list
    assert get_ipython().user_ns.get('_exit_code', 0) == 0, 'generation failed -- see traceback above'


In [ ]:
# Chatterbox writes its own manifest so the f5_tts_sw rows in fake_manifest.tsv are
# never overwritten. Merge both into one file for build_manifest.py.
import os
rows = []
for p in ('data/manifests/fake_manifest.tsv', 'data/manifests/fake_manifest_chatterbox.tsv'):
    if os.path.exists(p):
        with open(p, encoding='utf-8') as f:
            lines = f.read().splitlines()
        if not rows:
            rows.append(lines[0])
        rows.extend(l for l in lines[1:] if l.strip())
with open('data/manifests/fake_manifest_combined.tsv', 'w', encoding='utf-8') as f:
    f.write('\n'.join(rows) + '\n')
print('combined manifest rows:', len(rows) - 1)

# Verify the 1:1 count, then rebuild the combined labeled manifest.
import glob
n_fake = len(glob.glob("data/generated/chatterbox/*.wav"))
print(f"real reference clips : {N_TARGET}")
print(f"chatterbox deepfakes : {n_fake}")

!python scripts/build_manifest.py \
    data/manifests/selected_subset.present.tsv data/manifests/fake_manifest_combined.tsv \
    data/processed --output data/manifests/dataset_manifest.csv \
    --summary data/manifests/split_summary.json

## 10. Final sync

In [ ]:
if PUSH_RESULTS_TO_DRIVE:
    !rclone copy data/generated/chatterbox "{RCLONE_REMOTE}:{DRIVE_DATA_DIR}/generated/chatterbox" --transfers 32 --checkers 32 --fast-list
    !rclone copy data/manifests "{RCLONE_REMOTE}:{DRIVE_DATA_DIR}/manifests" --fast-list
    print("Final push to Drive done.")

print("Click 'Save Version' (Commit) to persist /kaggle/working "
      "(data/generated + data/manifests) as this notebook's output.")